# data_pipeline: Data pipelines for Pre-Training

## Learning Objectives
After studying and implementing this section, you should be able to:

1. Build a streaming data pipeline capable of tokenizing, chunking, shuffling, and batching terabytes of text without loading the entire dataset into memory.
2. Implement data quality filters such as deduplication, language detection, and content filtering, mirroring real-world pre-training pipelines.
3. Construct fixed-length training sequences, properly handling attention masks and document boundaries.
4. Measure and profile pipeline throughput to ensure the DataLoader feeds data fast enough to keep GPU compute fully utilized without starvation.


## What is the problem?
You have a tokenizer; now you need data.

This is not about a small dataset or a CSV file. To train a large language model, you need terabytes of text that has been cleaned, deduplicated, quality-filtered, and evaluated. Then, the text must be tokenized, divided into fixed-length sequences, and delivered to the model in random batches at sufficient speed. The pipeline's speed must be high enough that your eight-GPU cluster never has to wait for the next batch.

Many believe LLM training primarily depends on model architecture, but data plays a decisive role. Llama 3 was trained on 15.6 trillion tokens, GPT-3 on 300 billion tokens, and DeepSeek-V2 on 8.1 trillion tokens. The overall architecture of all three is more or less similar: multiple Transformer blocks stacked together, comprising attention and feed-forward layers. A significant portion of the difference in output quality among these models stems from the volume, composition, and quality of the training data.

DeepMind's Chinchilla paper elaborated on this. For any given compute budget, there is an optimal ratio between the number of model parameters and the number of training tokens. The research indicated that most models in 2022 were significantly undertrained, meaning they had too many parameters relative to the amount of data they were exposed to. For instance, a 70 billion parameter model trained on 1.4 trillion tokens, following the optimal Chinchilla ratio, outperformed Gopher, a 280 billion parameter model trained on only 300 billion tokens.

Therefore, your data pipeline determines whether the model truly learns language, knowledge, and useful patterns, or merely reproduces the noise present in the data.



In [2]:
import re
import hashlib
import random
import time
from collections import Counter, defaultdict
from typing import List, Tuple, Set, Dict, Any, Generator, Optional

In [ ]:
# basic clean text
def clean_text(text: str) -> str:
    """
    Cleans raw document text by stripping HTML tags, URLs, non-ASCII characters,
    and normalizing whitespace.

    Args:
        text (str): Input raw text document.

    Returns:
        str: Cleaned and normalized text string.
    """
    # TODO: Strip unwanted markup, non-ASCII noise, and normalize spaces/newlines.

    # if is NOT text-> return ""
    if not text:
        return ""

    # by regex remove tags
    text = re.sub(r"<[^>]+>", " ", text)

    # remove URL
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # remove ASCII
    text = re.sub(r"[^\x00-\x7F]+", " ", text)

    # remove extera space in first and end
    text = re.sub(r"\s+", " ", text).strip()

    return text



Implementing this function in the pre-training pipeline simulates data quality filtering, screening out low-quality inputs such as promotional pages or spam content.

This function must evaluate three criteria:
- Word count
- Ratio of all-uppercase words
- Special character density

This function originates from the logic of well-known pipelines like RefinedWeb and Falcon, which check thresholds for length, uppercase ratio, and punctuation ratio for each document. These three criteria are aggregated to ensure each document passes through three gates.



In [ ]:
def quality_filter(
    text: str, 
    min_words: int = 50, 
    max_ratio_caps: int = 0.3, 
    max_ratio_special: float = 0.1
) -> bool:
    """
    Filters out low-quality documents based on length, capitalization ratio, and special character density.

    Args:
        text (str): Cleaned document text.
        min_words (int): Minimum required word count.
        max_ratio_caps (float): Maximum allowed ratio of ALL-CAPS words.
        max_ratio_special (float): Maximum allowed ratio of non-alphanumeric special characters.

    Returns:
        bool: True if the document meets quality criteria, False otherwise.
    """
    # TODO: Check word count thresholds and measure capitalization and special-character ratios.
    
    # min vocab - Word count
    words = text.split()
    if len(words) < min_words:
        return False

    # Ratio of all-uppercase words
    caps_words = sum(
        1 for w in words
        if w.isalpha() and w == w.upper()
    )
    if caps_words / len(words) > max_ratio_caps:
        return False

    # Special character density
    n_special = sum(1 for ch in text if not ch.isalnum())
    if n_special / len(text) > max_ratio_special:
        return False

    return True
